# Wes's gaming + relevance eval

Two deliverables for the proposal's evaluation plan:

1. **Relevance LLM judge** — score every served ad on the 50-prompt benchmark; validate against a wz-labeled gold set with Cohen's κ.
2. **Gaming robustness** — three attacks × two defenses + naive baseline, reported as an inflation-delta matrix on the 3 × 4 grid `{no_defense, D_paraphrase, D_landing, D_both}`.

**Methodology disclosure (must appear in writeup).** wz hand-labeled all 50 items under a written rubric (`data/labeling_protocol.md`). Gemini 2.5 Pro independently labeled the same 50 as a cross-check, and disagreements were adjudicated by wz. The headline κ here is wz-vs-Sonnet judge agreement on the 43 served rows; the Gemini-vs-Sonnet number is reported as a secondary sanity check.

**Cost.** ~800 cached LLM calls (~$3–4 first run; $0 on re-runs via the disk cache).

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    %cd '/content/drive/Shared drives/[OIT 277] Final project/Eric code'
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') or ''
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY') or ''
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config.py').exists() and (PROJECT_ROOT.parent / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib.util
import subprocess
missing = [pkg for pkg in ('pandas', 'sklearn', 'anthropic', 'google.generativeai') if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])

import pandas as pd
from auction import relevance_judge, gold_labeling, gaming

assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY (Claude Sonnet judge + attack rewrites)'
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY (Gemini cross-check labeler)'

In [ ]:
defended = pd.read_csv(PROJECT_ROOT / 'results' / 'defended.csv')
wz_gold = pd.read_csv(PROJECT_ROOT / 'data' / 'gold_labels_relevance_wz.csv')
print(f'defended: {len(defended)} rows  ({defended["winner_advertiser_id"].notna().sum()} served)')
print(f'wz gold labels: {len(wz_gold)} rows  ({(wz_gold["wz_label"] != "N/A").sum()} scored, {(wz_gold["wz_label"] == "N/A").sum()} N/A)')

## 2. Run the LLM judge over the 50 honest served ads

Saves to `results/relevance_judge.csv`. Re-runs hit the cache and cost $0.

In [ ]:
judge_df = relevance_judge.judge_dataframe(defended)
(PROJECT_ROOT / 'results').mkdir(exist_ok=True)
judge_df.to_csv(PROJECT_ROOT / 'results' / 'relevance_judge.csv', index=False)
print(f"Saved {PROJECT_ROOT / 'results' / 'relevance_judge.csv'}")
judge_df.head(10)

## 3. Gemini cross-check + adjudication

In [ ]:
gemini_df = gold_labeling.gemini_crosscheck_labels(defended)
merged = gold_labeling.merge_with_wz_labels(wz_gold, gemini_df)
n_disagree = merged['needs_adjudication'].sum()
print(f'Adjudication needed on {n_disagree}/{len(merged)} rows.')
merged[merged['needs_adjudication']][['prompt_id', 'wz_label', 'gemini_label', 'wz_reasoning', 'gemini_reasoning']]

**Adjudication.** For each row flagged above, decide the final `gold_label` by reading both reasonings. The committed `gold_label` defaults to wz's primary label and is overwritten only where Gemini's reasoning is more compelling. Edit the dict below if any flips are needed, then re-run the next cell.

In [ ]:
# Adjudications baked in from the 2026-04-26 review of all 20 disagreements.
# Each entry records wz's deliberate call between the primary wz label and the Gemini cross-check.
# Re-running this cell reproduces data/gold_labels_relevance.csv exactly.
ADJUDICATIONS = {
    'p001': (0.5,  "kept wz: BlueParrott is a mono single-ear headset, real form-factor mismatch with 'headphones'."),
    'p003': (0.25, "took gemini: child-safety volume-limiter is THE central feature; recommending an adult ANC headphone for an 8-yr-old fails the central use case."),
    'p004': (0.75, "kept wz: Beats Pro is explicitly marketed as 'professional studio over-ear', closed-back; scoring by marketed positioning per protocol."),
    'p007': (0.75, "split: Surface Laptop is the right product class but ad copy doesn't actually confirm the all-day battery the user asked for; not a 1.0."),
    'p008': (0.75, "kept wz: 8GB RAM is genuinely limiting for modern PC gaming; partial use-case match."),
    'p009': (0.25, "took gemini: 6th-gen ULV i7 + HDD genuinely cannot do 1080p video editing well; central use case unmet."),
    'p012': (0.25, "took gemini: business projector misses ALL three central asks (portable, outdoor brightness, $500 budget)."),
    'p014': (0.25, "kept wz: still 'cameras' at the broadest sense per protocol's 0.25 definition."),
    'p016': (0.5,  "kept wz: rugged compact with cold/water marketing, but missing the central action-cam form factor (POV mounting)."),
    'p017': (0.5,  "took gemini: iPhone compatibility was an explicit user constraint; missing it is a meaningful miss."),
    'p022': (0.5,  "kept wz: Surface Pro 4 has the form factor but Core M + 4GB RAM in 2024 is too weak for serious illustration work."),
    'p029': (0.25, "took gemini: waterproof is the wrong feature focus when the user explicitly asked for drop-tested protection."),
    'p032': (0.0,  "took gemini: tablet-arm desk mount has no functional overlap with a mouse — wrong domain."),
    'p036': (0.0,  "took gemini: selling a 42\" TV when the user asked for a soundbar (and already has a 55\" TV) is wrong product entirely."),
    'p037': (0.25, "kept wz: speakers and turntables are different products but both in the audio/hi-fi domain."),
    'p039': (0.5,  "kept wz: CM500 functionally meets the asks (mic, over-ear, in budget) — gaming-specific positioning is more aesthetic than functional."),
    'p040': (0.0,  "took gemini: capture cards take video signals, PTZ cameras produce them — incompatible hardware, wrong domain."),
    'p041': (0.25, "took gemini: a sleep tracker tracks sleep, doesn't help you sleep — central use case unmet."),
    'p044': (0.0,  "took gemini: birding binoculars vs computer eye-strain reduction are different functional domains."),
    'p045': (0.5,  "split: over-ear is the right form factor for hearing-aid users, but ad makes no explicit hearing-aid-compatibility claim."),
}

def _commit(row):
    pid = row['prompt_id']
    if pid in ADJUDICATIONS:
        gold, note = ADJUDICATIONS[pid]
        return pd.Series({'gold_label': gold, 'adjudication_note': note})
    if row['needs_adjudication']:
        return pd.Series({
            'gold_label': row['wz_label'],
            'adjudication_note': f'NO ADJUDICATION RECORDED (gemini said {row["gemini_label"]})',
        })
    return pd.Series({'gold_label': row['wz_label'], 'adjudication_note': ''})

committed = merged.join(merged.apply(_commit, axis=1))
for col in ['wz_label', 'gemini_label', 'gold_label']:
    committed[col] = committed[col].fillna('N/A')
committed['labeler'] = 'wz+gemini_2.5_pro_crosscheck'
committed.to_csv(PROJECT_ROOT / 'data' / 'gold_labels_relevance.csv', index=False)
print(f"Saved {PROJECT_ROOT / 'data' / 'gold_labels_relevance.csv'} ({len(committed)} rows)")

n_adj = committed['adjudication_note'].astype(bool).sum()
print(f'Adjudicated rows: {n_adj}')
print(f'  kept wz:     {committed["adjudication_note"].str.startswith("kept wz", na=False).sum()}')
print(f'  took gemini: {committed["adjudication_note"].str.startswith("took gemini", na=False).sum()}')
print(f'  split:       {committed["adjudication_note"].str.startswith("split", na=False).sum()}')
committed[['prompt_id', 'wz_label', 'gemini_label', 'gold_label', 'adjudication_note']].head(15)


## 4. Three Cohen's κ values + Spearman vs embedding-relevance

- **wz-vs-Sonnet** — primary credibility number for the writeup (wz hand-labels vs the Sonnet judge).
- **Gemini-vs-Sonnet** — independent cross-check sanity number.
- **wz-vs-Gemini** — gold-set internal sanity check between the wz primary labels and the Gemini cross-check.

In [ ]:
# Build three (prompt_id, gold_label) frames so we can call the kappa helper three times.
def _gold_frame(series):
    return pd.DataFrame({'prompt_id': committed['prompt_id'],
                          'gold_label': pd.to_numeric(series.replace('N/A', None), errors='coerce')})

wz_as_gold  = _gold_frame(committed['wz_label'])
gem_as_gold = _gold_frame(committed['gemini_label'])
final_gold  = _gold_frame(committed['gold_label'])

kappa_wz    = relevance_judge.cohens_kappa_against_gold(judge_df, wz_as_gold)
kappa_gem   = relevance_judge.cohens_kappa_against_gold(judge_df, gem_as_gold)
kappa_final = relevance_judge.cohens_kappa_against_gold(judge_df, final_gold)

# wz-vs-Gemini sanity check (uses the kappa helper, treating wz as 'judge' and gemini as 'gold')
wz_judge_shape = pd.DataFrame({'id': committed['prompt_id'],
                                'judge_score': pd.to_numeric(committed['wz_label'].replace('N/A', None), errors='coerce')})
kappa_wz_vs_gem = relevance_judge.cohens_kappa_against_gold(wz_judge_shape, gem_as_gold)

spearman = relevance_judge.correlate_with_embedding(judge_df, defended)

print('=' * 70)
print('RELEVANCE JUDGE VALIDATION')
print('=' * 70)
print(f'wz-vs-Sonnet (primary credibility number):   κ={kappa_wz["kappa_unweighted"]:.3f} (linear-weighted {kappa_wz["kappa_linear"]:.3f}, n={kappa_wz["n_compared"]})')
print(f'Gemini-vs-Sonnet (cross-check sanity):       κ={kappa_gem["kappa_unweighted"]:.3f} (linear-weighted {kappa_gem["kappa_linear"]:.3f}, n={kappa_gem["n_compared"]})')
print(f'Final-gold-vs-Sonnet:                        κ={kappa_final["kappa_unweighted"]:.3f} (linear-weighted {kappa_final["kappa_linear"]:.3f}, n={kappa_final["n_compared"]})')
print(f'wz-vs-Gemini (gold-set sanity check):        κ={kappa_wz_vs_gem["kappa_unweighted"]:.3f} (n={kappa_wz_vs_gem["n_compared"]})')
print()
print(f'Spearman(judge, embedding relevance_score):  ρ={spearman["spearman"]:.3f}, n={spearman["n"]}')
print(f'Pearson(judge, embedding relevance_score):   r={spearman["pearson"]:.3f}')


## 5. Build the inflation-delta matrix

3 attacks × 4 defense columns. Each cell = mean over served prompts of
`defended_score(attacked_copy) - defended_score(honest_copy)`. A near-zero column = the defense neutralized that attack.

In [ ]:
matrix, full = gaming.build_inflation_matrix(defended)
matrix.to_csv(PROJECT_ROOT / 'results' / 'inflation_matrix.csv')
full.to_csv(PROJECT_ROOT / 'results' / 'inflation_full_records.csv', index=False)
print(f"\nSaved {PROJECT_ROOT / 'results' / 'inflation_matrix.csv'} and {PROJECT_ROOT / 'results' / 'inflation_full_records.csv'}")
print('\nMatrix:')
matrix.round(3)

In [ ]:
# Heatmap (optional; works if matplotlib is installed)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3))
    im = ax.imshow(matrix.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix.values[i, j]
            ax.text(j, i, f'{v:+.2f}' if pd.notna(v) else 'NA', ha='center', va='center',
                    color='white' if abs(v) > 0.3 else 'black')
    ax.set_title('Inflation-delta matrix: mean (attacked - honest) judge score')
    plt.colorbar(im, ax=ax, label='Δ score')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available; skip heatmap')

## 6. Writeup-ready summary block

Drop these numbers + caveats directly into the relevance/gaming sections of the writeup. Headline narrative is auto-generated below.

In [ ]:
lines = []
lines.append('## Relevance LLM judge')
lines.append(f'- Judged defended.csv with Sonnet 4.6 LLM judge: 43 served ads scored, 7 suppressed rows treated as N/A.')
lines.append(f'- Mean judge score: {pd.to_numeric(judge_df["judge_score"], errors="coerce").mean():.3f}')
lines.append(f'- κ vs wz primary labels (HEADLINE — primary credibility number): {kappa_wz["kappa_unweighted"]:.3f} ({kappa_wz["n_compared"]} compared rows)')
lines.append(f'- κ vs Gemini-cross-checked gold (independent cross-check sanity): {kappa_gem["kappa_unweighted"]:.3f}')
lines.append(f'- Spearman ρ with embedding relevance_score: {spearman["spearman"]:.3f} — positive but not redundant.')
lines.append('')
lines.append('## Gaming robustness (inflation-delta matrix)')
lines.append('Mean per-prompt inflation in judge score from honest → attacked copy, under each defense:')
lines.append('')
lines.append(matrix.round(3).to_string())
lines.append('')
lines.append('Interpretation guide:')
lines.append('- A column near 0 = defense neutralizes that attack.')
lines.append('- A1 (keyword stuffing) should pump no_defense and collapse under D_paraphrase.')
lines.append('- A2 (fabricated claims) should be reduced more by D_landing than by D_paraphrase.')
lines.append('- A3 (persona) should be partially defeated by D_both.')
lines.append('')
lines.append('## Methodology disclosure (REQUIRED in writeup)')
lines.append('- wz hand-labeled all 50 items under data/labeling_protocol.md; Gemini 2.5 Pro independent cross-check; wz adjudicated disagreements.')
lines.append('- Headline κ is wz-vs-Sonnet judge agreement on 43 served rows; Gemini-vs-Sonnet reported as a secondary sanity number.')
lines.append('- For a true two-rater κ, a teammate would need to add a second labeler column to the 50 rows.')
print('\n'.join(lines))